# 01 — Generate Customer & Order Data

Uses Faker to generate:
- **~200 customers** with unique IDs and emails
- **~500 orders** referencing valid customer IDs
- **~30 new_customers_batch** with deliberate duplicates (used later to trigger constraint violations)

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
from faker import Faker
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType, DateType
)
import os
import random

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

fake = Faker()
Faker.seed(42)
random.seed(42)

## Generate Customers

In [ ]:
NUM_CUSTOMERS = 200

customers = []
used_emails = set()

for i in range(1, NUM_CUSTOMERS + 1):
    # Ensure unique emails
    email = fake.email()
    while email in used_emails:
        email = fake.email()
    used_emails.add(email)

    customers.append((
        i,
        email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-3y", end_date="today"),
    ))

customer_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("email", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("signup_date", DateType(), True),
])

customers_df = spark.createDataFrame(customers, schema=customer_schema)
customers_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.customers")

print(f"Wrote {customers_df.count()} customers")
customers_df.show(5, truncate=False)

## Generate Orders

In [ ]:
NUM_ORDERS = 500

products = [
    "Laptop", "Keyboard", "Mouse", "Monitor", "Headphones",
    "Webcam", "USB Hub", "Desk Lamp", "Notebook", "Backpack",
]

orders = []
for i in range(1, NUM_ORDERS + 1):
    orders.append((
        i,
        random.randint(1, NUM_CUSTOMERS),
        random.choice(products),
        random.randint(1, 5),
        round(random.uniform(9.99, 499.99), 2),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

order_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", FloatType(), True),
    StructField("order_date", DateType(), True),
])

orders_df = spark.createDataFrame(orders, schema=order_schema)
orders_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.orders")

print(f"Wrote {orders_df.count()} orders")
orders_df.show(5, truncate=False)

## Generate New Customers Batch (with deliberate duplicates)

This batch has:
- **10 genuinely new** customers
- **10 with duplicate `customer_id`** (IDs that already exist in customers)
- **10 with duplicate `email`** (emails that already exist in customers)

In [ ]:
# Collect existing data for creating deliberate duplicates
existing_customers = spark.sql(
    f"SELECT customer_id, email FROM {UC_CATALOG}.{UC_SCHEMA}.customers"
).collect()

existing_ids = [row.customer_id for row in existing_customers]
existing_emails = [row.email for row in existing_customers]

batch = []

# 10 genuinely new customers (IDs starting after max existing)
max_id = max(existing_ids)
for i in range(1, 11):
    new_email = fake.email()
    while new_email in used_emails:
        new_email = fake.email()
    used_emails.add(new_email)

    batch.append((
        max_id + i,
        new_email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

# 10 with duplicate customer_id (pick random existing IDs, new emails)
dup_ids = random.sample(existing_ids, 10)
for dup_id in dup_ids:
    new_email = fake.email()
    while new_email in used_emails:
        new_email = fake.email()
    used_emails.add(new_email)

    batch.append((
        dup_id,  # duplicate customer_id!
        new_email,
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

# 10 with duplicate email (new IDs, but emails from existing customers)
dup_emails = random.sample(existing_emails, 10)
for dup_email in dup_emails:
    max_id += 11  # ensure unique IDs for this group
    batch.append((
        max_id,
        dup_email,  # duplicate email!
        fake.first_name(),
        fake.last_name(),
        fake.phone_number(),
        fake.city(),
        fake.state_abbr(),
        fake.date_between(start_date="-1y", end_date="today"),
    ))

batch_df = spark.createDataFrame(batch, schema=customer_schema)
batch_df.write.mode("overwrite").saveAsTable(f"{UC_CATALOG}.{UC_SCHEMA}.new_customers_batch")

print(f"Wrote {batch_df.count()} rows to new_customers_batch")
batch_df.show(30, truncate=False)

## Summary

In [ ]:
for table in ["customers", "orders", "new_customers_batch"]:
    count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {UC_CATALOG}.{UC_SCHEMA}.{table}").collect()[0].cnt
    print(f"{UC_CATALOG}.{UC_SCHEMA}.{table}: {count} rows")